# 🔍 RAG Pipeline
### `a.parquet` → Embeddings → Cosine Similarity Retrieval → LLM Answer

| Stage | What happens |
|---|---|
| **1. Load** | Read `a.parquet`, auto-detect text column |
| **2. Embed** | Encode all chunks with Sentence-Transformers |
| **3. Retrieve** | Cosine similarity search for top-K chunks |
| **4. Generate** | Claude reads context and answers the query |

## ⚙️ Install Dependencies

In [ ]:
!pip install pandas pyarrow sentence-transformers anthropic numpy scikit-learn --quiet

## 📦 Imports

In [ ]:
import os
import numpy as np
import pandas as pd
from typing import Optional
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import anthropic

print("✅ All imports successful")

## 🔧 Configuration
> Edit these values before running the pipeline.

In [ ]:
PARQUET_PATH = "a.parquet"          # Path to your dataset
TEXT_COLUMN  = None                 # e.g. "text" — None = auto-detect
TOP_K        = 5                    # Number of chunks to retrieve
EMBED_MODEL  = "all-MiniLM-L6-v2"  # Sentence-Transformers model
LLM_MODEL    = "claude-sonnet-4-20250514"
MAX_TOKENS   = 1024

# Set your Anthropic API key here OR via environment variable
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."

print(f"Parquet : {PARQUET_PATH}")
print(f"Top-K   : {TOP_K}")
print(f"Embedder: {EMBED_MODEL}")
print(f"LLM     : {LLM_MODEL}")

---
## Stage 1 — 📂 Load Data

In [ ]:
def load_parquet(path: str, text_col: Optional[str] = None):
    """Load parquet file and extract text chunks."""
    print(f"Loading '{path}'...")
    df = pd.read_parquet(path)
    print(f"  Rows: {len(df)} | Columns: {list(df.columns)}")

    if text_col is None:
        candidates = [c for c in df.columns if any(
            kw in c.lower() for kw in ["text", "content", "body", "desc", "summary", "doc"]
        )]
        if candidates:
            text_col = candidates[0]
            print(f"  Auto-detected text column: '{text_col}'")
        else:
            str_cols = [c for c in df.columns if df[c].dtype == object]
            if not str_cols:
                raise ValueError("No string columns found. Set TEXT_COLUMN manually.")
            text_col = str_cols[0]
            print(f"  Falling back to first string column: '{text_col}'")

    chunks = df[text_col].dropna().astype(str).tolist()
    print(f"  Extracted {len(chunks)} text chunks.")
    return chunks, df, text_col


chunks, df, detected_col = load_parquet(PARQUET_PATH, TEXT_COLUMN)

# Preview
print("\n── DataFrame Preview ──")
df.head(3)

In [ ]:
# Preview first 3 chunks
print("── First 3 Chunks ──")
for i, c in enumerate(chunks[:3]):
    print(f"[{i}] {c[:200]}\n")

---
## Stage 2 — 🔢 Generate Embeddings

In [ ]:
def embed_chunks(chunks, model_name):
    """Generate sentence embeddings for all chunks."""
    print(f"Loading embedding model '{model_name}'...")
    model = SentenceTransformer(model_name)
    print(f"Encoding {len(chunks)} chunks...")
    embeddings = model.encode(chunks, show_progress_bar=True, batch_size=64)
    print(f"\nEmbedding matrix: {embeddings.shape}  (chunks × dimensions)")
    return embeddings, model


chunk_embeddings, embed_model = embed_chunks(chunks, EMBED_MODEL)

In [ ]:
# Sanity check — embedding stats
print(f"Shape  : {chunk_embeddings.shape}")
print(f"Min    : {chunk_embeddings.min():.4f}")
print(f"Max    : {chunk_embeddings.max():.4f}")
print(f"Mean   : {chunk_embeddings.mean():.4f}")
print(f"Norm[0]: {np.linalg.norm(chunk_embeddings[0]):.4f}  (should be ~1.0 for normalised models)")

---
## Stage 3 — 📐 Cosine Similarity Retrieval

In [ ]:
def retrieve(query, chunks, chunk_embeddings, embed_model, top_k=TOP_K):
    """Embed query and retrieve top-k chunks by cosine similarity."""
    query_embedding = embed_model.encode([query])                        # shape: (1, dim)
    scores = cosine_similarity(query_embedding, chunk_embeddings)[0]     # shape: (n_chunks,)

    top_indices = np.argsort(scores)[::-1][:top_k]
    results = [
        {"rank": i + 1, "score": float(scores[idx]), "chunk": chunks[idx], "index": int(idx)}
        for i, idx in enumerate(top_indices)
    ]
    return results, scores


# ── Try a query ──
QUERY = "What are the main topics in this dataset?"   # ← change me

results, all_scores = retrieve(QUERY, chunks, chunk_embeddings, embed_model, TOP_K)

print(f"Query: {QUERY}\n")
for r in results:
    print(f"  #{r['rank']}  score={r['score']:.4f}  |  {r['chunk'][:120]}")

In [ ]:
# Visualise the score distribution
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Histogram of all scores
axes[0].hist(all_scores, bins=40, color="steelblue", edgecolor="white")
axes[0].set_title("Cosine Similarity Distribution (all chunks)")
axes[0].set_xlabel("Score")
axes[0].set_ylabel("Count")

# Bar chart of top-K scores
ranks  = [f"#{r['rank']}" for r in results]
scores = [r['score'] for r in results]
bars = axes[1].bar(ranks, scores, color="coral", edgecolor="white")
axes[1].set_title(f"Top-{TOP_K} Retrieved Chunks")
axes[1].set_xlabel("Rank")
axes[1].set_ylabel("Cosine Similarity")
axes[1].set_ylim(0, 1)
for bar, s in zip(bars, scores):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f"{s:.3f}", ha="center", fontsize=9)

plt.tight_layout()
plt.show()

---
## Stage 4 — 🤖 LLM Generation (Claude)

In [ ]:
def generate(query, retrieved):
    """Feed retrieved context + query to Claude and return grounded answer."""
    context_block = "\n\n".join(
        f"[Chunk {r['rank']} | similarity={r['score']:.4f}]\n{r['chunk']}"
        for r in retrieved
    )

    system_prompt = (
        "You are a precise question-answering assistant. "
        "Answer the user's question using ONLY the provided context chunks. "
        "If the answer cannot be found in the context, say so clearly. "
        "Cite which chunk(s) support your answer."
    )

    user_message = (
        f"Context:\n{context_block}\n\n"
        f"Question: {query}\n\n"
        "Answer (with chunk citations):"
    )

    client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))
    response = client.messages.create(
        model=LLM_MODEL,
        max_tokens=MAX_TOKENS,
        system=system_prompt,
        messages=[{"role": "user", "content": user_message}],
    )
    return response.content[0].text


answer = generate(QUERY, results)

print("=" * 60)
print("QUESTION:", QUERY)
print("=" * 60)
print("ANSWER:\n")
print(answer)

---
## 🔁 Full Pipeline Class
> Wraps all 4 stages. Build once, query many times.

In [ ]:
class RAGPipeline:
    def __init__(self, parquet_path=PARQUET_PATH, text_col=TEXT_COLUMN):
        self.chunks, self.df, _ = load_parquet(parquet_path, text_col)
        self.chunk_embeddings, self.embed_model = embed_chunks(self.chunks, EMBED_MODEL)
        print("\n✅ RAG pipeline ready.")

    def query(self, question, top_k=TOP_K):
        """Full RAG: retrieve then generate."""
        retrieved, _ = retrieve(question, self.chunks, self.chunk_embeddings, self.embed_model, top_k)
        answer = generate(question, retrieved)
        return {"question": question, "retrieved": retrieved, "answer": answer}

    def semantic_search(self, question, top_k=TOP_K):
        """Retrieval only — no LLM call."""
        retrieved, _ = retrieve(question, self.chunks, self.chunk_embeddings, self.embed_model, top_k)
        return retrieved


# Build (reuses already-loaded data if chunks/embeddings exist)
rag = RAGPipeline()

---
## 💬 Interactive Query Cell
> Change `MY_QUESTION` and re-run this cell anytime.

In [ ]:
MY_QUESTION = "What are the main topics in this dataset?"  # ← edit me

result = rag.query(MY_QUESTION)

print("=" * 60)
print("❓ QUESTION:", result["question"])
print("=" * 60)
print("🤖 ANSWER:\n")
print(result["answer"])
print("\n" + "=" * 60)
print("📄 RETRIEVED CHUNKS:")
for r in result["retrieved"]:
    print(f"  #{r['rank']}  score={r['score']:.4f}  |  {r['chunk'][:120]}")